- This code runs 5 independent base queries for seaching 3 languages and the keyword android once in the topic and once in the name, desc and readme
- All 5 queries includes three common qualifier: stars:>50, fork:false and archived:false
- Layer 1: Live Search - GitHub API
- Layer 2: Local Filter - real-time Json (double Check the API detections)


In [ ]:
import os
import re
import requests
import pandas as pd
import logging
from dotenv import load_dotenv
from datetime import datetime, timedelta
from time import sleep, time as now


# === LOGGING ===
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# === CONFIG ===
output_dir = r"C:\Android Mobile App\Step1_URL_Search"
output_csv_all = os.path.join(output_dir, "github_android_search_results_all.csv")
output_csv_filtered = os.path.join(output_dir, "github_android_search_results_filtered.csv")

os.makedirs(output_dir, exist_ok=True)

# === AUTH ===
load_dotenv("All_Tokens.env")
tokens = [v for k, v in os.environ.items() if k.startswith("GITHUB_TOKEN_") and v]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

token_index = 0
HEADERS = {
    "Authorization": f"token {tokens[token_index]}",
    "Accept": "application/vnd.github+json",
    "User-Agent": "android-repo-crawler/1.0"
}

# === DATE RANGE ===
start_date = datetime.strptime("2008-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2024-12-31", "%Y-%m-%d")

# === WINDOW SETTINGS ===
initial_window_hours = 15 * 24  # 10 days
min_window_hours = 1
max_window_hours = 90 * 24
MAX_RESULTS_PER_QUERY = 1000
TARGET_FILL_RATIO = 0.25

# === BASE QUERIES ===
# Basis fully independent search for languages and keywords 
base_queries = [
    "stars:>50 language:Kotlin fork:false archived:false",
    "stars:>50 language:Java fork:false archived:false",
    "stars:>50 language:Dart fork:false archived:false",
    "stars:>50 topic:android fork:false archived:false",
    "stars:>50 android in:name,description,readme fork:false archived:false",
]


# === Filter ===
rejected_items = []  # To store items returned by GitHub but filtered out

def filter_item(item, expected_fork, expected_archived, expected_language, expected_topic, expected_keyword):
    min_stars = 50
    passed = True  # Flag to determine if the item passes all checks
    rejection_reasons = []

    language = (item.get('language') or '').lower()
    topics = [t.lower() for t in item.get('topics', [])]
    description = (item.get('description') or '').lower()
    name = (item.get('name') or '').lower()
    full_name = item.get("full_name", "")
    
    if item.get('stargazers_count', 0) <= min_stars:
        passed = False
        rejection_reasons.append("Below minimum stars")
    if item.get('fork', False) != expected_fork:
        passed = False
        rejection_reasons.append("Fork status mismatch")
    if item.get('archived', False) != expected_archived:
        passed = False
        rejection_reasons.append("Archived status mismatch")

    if expected_language and language != expected_language:
        passed = False
        rejection_reasons.append(f"Language mismatch (expected {expected_language})")
    if expected_topic and expected_topic not in topics:
        passed = False
        rejection_reasons.append("Missing topic: android")

    if expected_keyword:
        keyword = expected_keyword.lower()
        if keyword not in name and keyword not in description:
            # Check README only if name/description don’t contain keyword
            try:
                owner, repo = full_name.split("/")
                readme_url = f"https://api.github.com/repos/{owner}/{repo}/readme"
                readme_headers = {
                    "Authorization": f"token {tokens[token_index]}",
                    "Accept": "application/vnd.github.v3.raw",
                    "User-Agent": "android-repo-crawler/1.0"
                }
                r = requests.get(readme_url, headers=readme_headers, timeout=10)
                if r.status_code == 200:
                    readme_text = r.text.lower()
                    if not re.search(rf"\b{re.escape(keyword)}\b", readme_text, re.IGNORECASE):
                        passed = False
                        rejection_reasons.append(f"Keyword '{keyword}' not found in README (case-insensitive)")
                else:
                    logger.warning(f"❌ README fetch failed ({r.status_code}) for {full_name}")
                    passed = False
            except Exception as e:
                #logger.warning(f"⚠️ Exception fetching README for {full_name}: {e}")
                passed = False
                rejection_reasons.append(f"README fetch exception: {str(e)}")

    # Track rejected items
    if not passed:
        rejected_items.append({
            "full_name": full_name,
            "language": item.get("language"),
            "description": item.get("description"),
            "topics": item.get("topics"),
            "stars": item.get("stargazers_count"),
            "fork": item.get("fork"),
            "archived": item.get("archived"),
            "base_qualifier": item.get("base_qualifier", "N/A"),
            "html_url": item.get("html_url"),
            "rejection_reasons": "; ".join(rejection_reasons)
        })

    return passed



# === Token Rotate ===
def rotate_token():
    global token_index, HEADERS
    token_index = (token_index + 1) % len(tokens)
    HEADERS = {
        "Authorization": f"token {tokens[token_index]}",
        "Accept": "application/vnd.github+json"
    }
    logger.warning(f"🔑 Rotated → Token #{token_index + 1} / {len(tokens)}")

# === Rate Limit ===
def check_rate_limit():
    r = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
    if r.status_code != 200:
        logger.warning("⚠️  Could not check rate limit.")
        return
    data = r.json()
    remaining = data['resources']['search']['remaining']
    reset_epoch = data['resources']['search']['reset']
    reset_in = max(0, reset_epoch - now())
    logger.info(f"🔎 Remaining: {remaining} | Resets in {reset_in/60:.1f} min")
    if remaining < 5:
        if len(tokens) > 1:
            rotate_token()
            check_rate_limit()
        else:
            logger.warning(f"⏳ No extra tokens → sleeping {reset_in/60:.1f} min")
            sleep(reset_in + 5)

# === Count ===
def check_count(query):
    while True:
        check_rate_limit()
        r = requests.get("https://api.github.com/search/repositories",
                         headers=HEADERS, params={"q": query, "per_page": 1})
        if r.status_code == 200:
            return r.json().get("total_count", 0)
        elif r.status_code == 403:
            rotate_token()
        else:
            logger.error(f"❌ Count error: {r.status_code} — {r.text}")
            return -1

# === Fetch ===
def fetch_items(query):
    all_items = []
    for page in range(1, 11):
        check_rate_limit()
        r = requests.get("https://api.github.com/search/repositories",
                         headers=HEADERS, params={"q": query, "per_page": 100, "page": page})
        if r.status_code == 200:
            items = r.json().get("items", [])
            if not items:
                break
            all_items.extend(items)
            sleep(1)
        elif r.status_code == 403:
            rotate_token()
            return fetch_items(query)
        else:
            logger.error(f"❌ Fetch error: {r.status_code} — {r.text}")
            break
    return all_items

# === MAIN ===
final_results = []
for base_query_prefix in base_queries:
    current_start = start_date
    window_hours = initial_window_hours

    while current_start < end_date:
        current_end = min(current_start + timedelta(hours=window_hours), end_date)
        date_range = f"created:{current_start.isoformat()}..{current_end.isoformat()}"
        base_query = f"{base_query_prefix} {date_range}"

        total_count = check_count(base_query)
        logger.info(f"⏳ {base_query} → {total_count} repos")

        if total_count >= MAX_RESULTS_PER_QUERY and window_hours > min_window_hours:
            window_hours = max(window_hours // 2, min_window_hours)
            continue
        elif total_count < MAX_RESULTS_PER_QUERY * TARGET_FILL_RATIO and window_hours * 2 <= max_window_hours:
            window_hours = min(window_hours * 2, max_window_hours)

        items = fetch_items(base_query)
        expected_fork = 'fork:true' in base_query_prefix
        expected_archived = 'archived:true' in base_query_prefix
        expected_language = ''
        expected_topic = ''
        expected_keyword = ''

        if 'language:Kotlin' in base_query_prefix:
            expected_language = 'kotlin'
        elif 'language:Java' in base_query_prefix:
            expected_language = 'java'
        elif 'language:Dart' in base_query_prefix:
            expected_language = 'dart'
        elif 'topic:android' in base_query_prefix:
            expected_topic = 'android'
        elif 'android' in base_query_prefix:
            expected_keyword = 'android'


        count_passed = 0  # ✅ NEW counter for this window
        for item in items:
            # if filter_item(item, expected_fork, expected_archived, expected_language, expected_topic, expected_keyword):
            #     # === Check for valid clone_url ===
            #     clone_url = item.get("clone_url", "").strip()
            #     if not clone_url:
            #         logger.warning(f"⚠️ Skipped: missing or blank clone_url for {item.get('full_name')}")
            #         continue  # skip this one!
            item["base_qualifier"] = base_query_prefix
            if filter_item(item, expected_fork, expected_archived, expected_language, expected_topic, expected_keyword):
                clean_item = {
                    "name": item.get("name"),
                    "full_name": item.get("full_name"),
                    "language": item.get("language"),
                    "stargazers_count": item.get("stargazers_count"),
                    "forks": item.get("forks"),
                    "topics": item.get("topics"),
                    #"description": item.get("description"),
                    "fork": item.get("fork"),
                    "archived": item.get("archived"),
                    "owner.login": item.get("owner", {}).get("login"),
                    "html_url": item.get("html_url"),
                    "clone_url": item.get("clone_url"),
                    "visibility": item.get("visibility"),
                    "size": item.get("size"),
                    "open_issues_count": item.get("open_issues_count"),
                    "base_qualifier": base_query_prefix,
                    "search_qualifier": base_query,
                    "repo_stars": item.get("stargazers_count", 0),
                    "match_type": expected_language.capitalize() or expected_topic.capitalize() or expected_keyword.capitalize()
                }
                final_results.append(clean_item)
                count_passed += 1  # ✅ increment

        logger.info(f"✅ Window done: {count_passed} items")


        current_start = current_end + timedelta(seconds=1)


# ✅ === SAVE CLEAN CSV/XLSX ===

# Optional: create unique timestamp to prevent overwriting
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1️⃣ One final DataFrame: from CLEAN all_results
df_clean = pd.DataFrame(final_results)

# 2️⃣ Quick check: show a preview to be sure it's clean
print("📌 Columns:", list(df_clean.columns))
print(df_clean.head(5))

# 3️⃣ Choose output file names — timestamped for safety
output_csv = os.path.join(output_dir, f"search_results_{ts}.csv")
output_xlsx = os.path.join(output_dir, f"search_results_{ts}.xlsx")


# 4️⃣ Save to CSV and Excel
df_clean.to_csv(output_csv, index=False)
df_clean.to_excel(output_xlsx, index=False)

logger.info(f"✅ All done! Clean results saved to: {output_csv} and {output_xlsx}")

# 5️⃣ Optional: show final row count
print(f"✅ Final rows saved: {len(df_clean)}")

# === Save Rejected Items ===
if rejected_items:
    df_rejected = pd.DataFrame(rejected_items)
    rejected_csv_path = os.path.join(output_dir, f"rejected_repos_{ts}.csv")
    df_rejected.to_csv(rejected_csv_path, index=False)
    logger.info(f"🚫 Rejected repos saved to: {rejected_csv_path}")
else:
    logger.info("✅ No rejected repos to save.")





[INFO] 🔎 Remaining: 20 | Resets in 0.1 min
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-01-01T00:00:00..2008-01-16T00:00:00 → 0 repos
[INFO] 🔎 Remaining: 19 | Resets in 0.1 min
[INFO] ✅ Window done: 0 items
[INFO] 🔎 Remaining: 18 | Resets in 0.1 min
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-01-16T00:00:01..2008-02-15T00:00:01 → 0 repos
[INFO] 🔎 Remaining: 17 | Resets in 0.1 min
[INFO] ✅ Window done: 0 items
[INFO] 🔎 Remaining: 16 | Resets in 0.1 min
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-02-15T00:00:02..2008-04-15T00:00:02 → 0 repos
[INFO] 🔎 Remaining: 15 | Resets in 0.1 min
[INFO] ✅ Window done: 0 items
[INFO] 🔎 Remaining: 14 | Resets in 0.1 min
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-04-15T00:00:03..2008-06-14T00:00:03 → 0 repos
[INFO] 🔎 Remaining: 13 | Resets in 0.1 min
[INFO] ✅ Window done: 0 items
[INFO] 🔎 Remaining: 12 | Resets in 0.1 min
[INFO] ⏳ star

📌 Columns: ['name', 'full_name', 'language', 'stargazers_count', 'forks', 'topics', 'fork', 'archived', 'owner.login', 'html_url', 'clone_url', 'visibility', 'size', 'open_issues_count', 'base_qualifier', 'search_qualifier', 'repo_stars', 'match_type']
                   name                      full_name language  \
0         quran_android            quran/quran_android   Kotlin   
1            gobandroid                ligi/gobandroid   Kotlin   
2               javabot             evanchooly/javabot   Kotlin   
3  facebook-android-sdk  facebook/facebook-android-sdk   Kotlin   
4               tnoodle                 thewca/tnoodle   Kotlin   

   stargazers_count  forks                                             topics  \
0              2173    918                                   [android, quran]   
1               237     66  [android, android-app, goban, kotlin, kotlin-a...   
2                56     30                                                 []   
3              6256 

[INFO] ✅ All done! Clean results saved to: C:\Android Mobile App\Step1_URL_Search\search_results_20250626_003510.csv and C:\Android Mobile App\Step1_URL_Search\search_results_20250626_003510.xlsx
[INFO] 🚫 Rejected repos saved to: C:\Android Mobile App\Step1_URL_Search\rejected_repos_20250626_003510.csv


✅ Final rows saved: 89839
